In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel ,Field
from langchain.schema.runnable import RunnableParallel,RunnableBranch,RunnableLambda,RunnableSequence,RunnableLambda
from typing import Literal
load_dotenv()

True

In [3]:
prompt1 = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=["topic"]
)

In [4]:
prompt2 = PromptTemplate(
    template="Summarize the following text \n {text}",
    input_variables=["text"]
)

In [5]:
# Model
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [6]:
parser = StrOutputParser()

In [7]:
detail_report_chain = RunnableSequence(prompt1,model,parser)

In [8]:
branch_chain =RunnableBranch(
    (lambda x : len(x.split()) > 500, RunnableSequence(prompt2,model,parser)), 
    detail_report_chain
)

In [9]:
final_chain = RunnableSequence(detail_report_chain,branch_chain)

In [10]:
final_chain.invoke({"topic":"CSK vs RCB in 2025"})

"In a hypothetical 2025 IPL match, CSK defeated RCB by 7 wickets in Chennai. RCB scored 185/6, with contributions from Kohli and Maxwell Jr. CSK chased it down, led by a century from Aryan Sharma and support from Suresh Raina Jr. The match featured sons and grandsons of legendary players, highlighting the next generation of cricketers. Sharma was named Player of the Match. Key talking points included CSK's home dominance, RCB's bowling concerns, and Kohli's farewell season. The report also includes hypothetical squads for both teams."

In [11]:
final_chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
